In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import gradio as gr
import re

# ==== CONFIG ====
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = './data/f1_model_final_v3.pth'
CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
NUM_COLS = [
    'year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
    'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
    'Driver_Prev_Season_FL_Count', 'Is_Home_Race'
]
TARGET_COL = 'is_winner'

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
os.makedirs('./data', exist_ok=True)

# ==== HELPER ====
def clean_string(text):
    if isinstance(text, str):
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

GP_COUNTRY_MAP = {
    'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
    'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
    'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
    'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
    'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
    'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
}
def get_country_from_gp(gp_name):
    if not isinstance(gp_name, str): return None
    for key, country_code in GP_COUNTRY_MAP.items():
        if key in gp_name: return country_code
    return None

def fill_missing(df, cat_cols, num_cols):
    for col in cat_cols:
        if col not in df.columns: df[col] = 'Unknown'
        df[col] = df[col].fillna('Unknown')
    for col in num_cols:
        if col not in df.columns: df[col] = 0
        df[col] = df[col].fillna(0)
    return df

# ==== 1. DATA LOADING & CLEANING ====
def load_and_clean_data(exclude_indy=True):
    files = {'winners': 'winners.csv', 'drivers': 'drivers_updated.csv',
             'teams': 'teams_updated.csv', 'fastest_laps': 'fastest_laps_updated.csv'}
    df_map = {k: pd.read_csv(f'./data/{v}', encoding='utf-8') for k, v in files.items()}
    for df in df_map.values():
        for c in df.select_dtypes(include='object'): df[c] = df[c].astype(str).str.strip()
    df_map['winners']['year'] = pd.to_datetime(df_map['winners']['Date'], errors='coerce').dt.year.astype(int)
    df_map['drivers'].rename(columns={'Car': 'Team'}, inplace=True)
    for n in ['drivers', 'teams', 'fastest_laps']:
        df = df_map[n]
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
        if 'Pos' in df.columns: df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
    if exclude_indy:
        for n in ['winners', 'fastest_laps']:
            df_map[n] = df_map[n][~df_map[n]['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    return df_map['winners'], df_map['drivers'], df_map['teams'], df_map['fastest_laps']

# ==== 2. FEATURE ENGINEERING ====
def engineer_features(drivers_df, teams_df, fastest_laps_df, default_prev_pos_driver, default_prev_pos_team):
    # Driver lagged features
    drivers_df = drivers_df.sort_values(['Driver', 'year'])
    drivers_df['Prev_Year_Driver_PTS'] = drivers_df.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers_df['Prev_Year_Driver_Pos'] = drivers_df.groupby('Driver')['Pos'].shift(1).fillna(default_prev_pos_driver)
    drivers_df['Driver_Experience_Years'] = drivers_df['year'] - drivers_df.groupby('Driver')['year'].transform('min')
    # Team lagged features
    teams_df = teams_df.sort_values(['Team', 'year'])
    teams_df['Prev_Year_Team_PTS'] = teams_df.groupby('Team')['PTS'].shift(1).fillna(0)
    teams_df['Prev_Year_Team_Pos'] = teams_df.groupby('Team')['Pos'].shift(1).fillna(default_prev_pos_team)
    teams_df['Team_Experience_Years'] = teams_df['year'] - teams_df.groupby('Team')['year'].transform('min')
    # Merge team to drivers
    drivers_df = drivers_df.merge(
        teams_df[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']],
        on=['Team', 'year'], how='left').fillna(0)
    # Fastest lap lagged feature
    fl = fastest_laps_df.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
    fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
    drivers_df = drivers_df.merge(fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']],
                                 on=['Driver', 'year'], how='left').fillna(0)
    return drivers_df

# ==== 3. BUILD MODEL DATA ====
def restructure_for_modeling(winners_df, drivers_df, default_prev_pos_driver, default_prev_pos_team):
    samples = []
    drivers_by_year = {y: g for y, g in drivers_df.groupby('year')}
    for _, race in winners_df.iterrows():
        year, gp, winner = race['year'], race['Grand Prix'], race['Winner']
        race_country = get_country_from_gp(gp)
        if year not in drivers_by_year: continue
        for _, drv in drivers_by_year[year].iterrows():
            is_home = 1 if race_country and drv['Nationality'] == race_country else 0
            samples.append({
                'year': year, 'Grand Prix': gp, 'Driver': drv['Driver'],
                'Team': drv['Team'], 'Nationality': drv['Nationality'],
                'Prev_Year_Driver_PTS': drv['Prev_Year_Driver_PTS'],
                'Prev_Year_Driver_Pos': drv['Prev_Year_Driver_Pos'],
                'Driver_Experience_Years': drv['Driver_Experience_Years'],
                'Prev_Year_Team_PTS': drv['Prev_Year_Team_PTS'],
                'Prev_Year_Team_Pos': drv['Prev_Year_Team_Pos'],
                'Team_Experience_Years': drv['Team_Experience_Years'],
                'Driver_Prev_Season_FL_Count': drv['Driver_Prev_Season_FL_Count'],
                'Is_Home_Race': is_home,
                'is_winner': int(drv['Driver'] == winner)
            })
    return pd.DataFrame(samples)

# ==== 4. PREPROCESSING (ENCODING & SCALING) ====
def preprocess_data(train_df, test_df, cat_cols, num_cols):
    label_encoders, cat_feat_dims = {}, {}
    for col in cat_cols:
        le = LabelEncoder()
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)
        le.fit(train_df[col])
        train_df[col] = le.transform(train_df[col])
        test_df[col] = test_df[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else len(le.classes_))
        label_encoders[col] = le
        cat_feat_dims[col] = len(le.classes_) + 1
    scaler = StandardScaler()
    train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
    test_df[num_cols] = scaler.transform(test_df[num_cols])
    return train_df, test_df, label_encoders, scaler, cat_feat_dims, cat_cols, num_cols

# ==== 5. DATASET & DATALOADER ====
class F1Dataset(Dataset):
    def __init__(self, df, cat_cols, num_cols, target_col):
        self.X_cat = df[cat_cols].values
        self.X_num = df[num_cols].values
        self.y = df[target_col].values
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X_cat[idx], dtype=torch.long), \
               torch.tensor(self.X_num[idx], dtype=torch.float32), \
               torch.tensor(self.y[idx], dtype=torch.float32)

# ==== 6. MODEL ====
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_feats, emb_dim=32, hidden_dim=128, dropout_rate=0.4):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_num_feats)
        self.fc = nn.Sequential(
            nn.Linear(len(cat_dims)*emb_dim + num_num_feats, hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim//2, 1)
        )
    def forward(self, x_cat, x_num):
        x = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        x = torch.cat([x, self.bn_num(x_num)], 1)
        return self.fc(x)

# ==== 7. TRAINING ====
def train_model(model, trainloader, testloader, n_epoch=30, lr=5e-4, patience=7, model_path=MODEL_PATH):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    best_test_loss, no_improve_epochs = np.inf, 0
    train_losses, test_losses = [], []
    for epoch in range(n_epoch):
        model.train()
        epoch_train_loss = 0
        for x_cat, x_num, y_true in trainloader:
            x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_cat, x_num), y_true.unsqueeze(1))
            loss.backward(); optimizer.step()
            epoch_train_loss += loss.item()
        avg_train_loss = epoch_train_loss / len(trainloader)
        train_losses.append(avg_train_loss)
        # Eval
        model.eval(); epoch_test_loss = 0
        with torch.no_grad():
            for x_cat, x_num, y_true in testloader:
                x_cat, x_num, y_true = x_cat.to(DEVICE), x_num.to(DEVICE), y_true.to(DEVICE)
                loss = criterion(model(x_cat, x_num), y_true.unsqueeze(1))
                epoch_test_loss += loss.item()
        avg_test_loss = epoch_test_loss / len(testloader)
        test_losses.append(avg_test_loss)
        print(f"Epoch {epoch+1}/{n_epoch} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), model_path)
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience: print("Early stopping."); break
    # Loss curve
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,6)); plt.plot(train_losses,label='Train'); plt.plot(test_losses,label='Test')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curve'); plt.legend(); plt.grid(True)
    plt.savefig('./data/loss_curve_final.png'); plt.close()
    return train_losses, test_losses

# ==== 8. EVALUATION ====
def evaluate_model(model, dataloader, model_path=MODEL_PATH):
    if model_path and os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval(); all_y_true, all_y_pred_classes = [], []
    with torch.no_grad():
        for x_cat, x_num, y_true in dataloader:
            x_cat,x_num,y_true=x_cat.to(DEVICE),x_num.to(DEVICE),y_true.to(DEVICE)
            logits = model(x_cat, x_num); probs = torch.sigmoid(logits)
            preds = (probs > 0.5).squeeze().int()
            all_y_true.extend(y_true.cpu().numpy())
            all_y_pred_classes.extend(preds.cpu().numpy() if preds.ndim > 0 else [preds.item()])
    print(f"\nAccuracy: {accuracy_score(all_y_true, all_y_pred_classes):.4f}")
    print(classification_report(all_y_true, all_y_pred_classes, target_names=['Not Winner(0)','Winner(1)'], zero_division=0))
    print(f"Confusion Matrix:\n{confusion_matrix(all_y_true, all_y_pred_classes)}")

# ==== 9. GRADIO PREDICT ====
GRADIO_VARS = dict(label_encoders={}, scaler=None, model=None, driver_info_by_year={},
                   all_years=[], all_gps=[], cat_cols_ordered=[], num_cols_ordered=[],
                   default_prev_pos_driver=50, default_prev_pos_team=20)

def gradio_predict_winner_probabilities(year_input, grand_prix_input):
    gv = GRADIO_VARS
    if not gv["model"] or not gv["label_encoders"] or (gv["num_cols_ordered"] and gv["scaler"] is None):
        return "Error: Model or preprocessors not loaded."
    try: year = int(year_input)
    except: return "Error: Invalid year."
    if not grand_prix_input: return "Error: Grand Prix input empty."
    drivers = gv["driver_info_by_year"].get(year, [])
    if not drivers: return f"No driver data for {year}."
    model = gv["model"]; model.eval(); res = []
    for p in drivers:
        cat = [gv["label_encoders"][col].transform([str(p.get(col, 'Unknown'))])[0] 
               if str(p.get(col, '')) in set(gv["label_encoders"][col].classes_)
               else len(gv["label_encoders"][col].classes_) for col in gv["cat_cols_ordered"]]
        num = [float(p.get(col, 0)) if col != 'Is_Home_Race' else
               1.0 if get_country_from_gp(grand_prix_input) == p.get('Nationality') else 0.0
               for col in gv["num_cols_ordered"]]
        x_cat = torch.tensor([cat], dtype=torch.long).to(DEVICE)
        x_num = torch.tensor(gv["scaler"].transform([num]), dtype=torch.float32).to(DEVICE) if num else torch.empty(1,0)
        with torch.no_grad():
            prob = torch.sigmoid(model(x_cat, x_num)).cpu().item()
        res.append((p.get('Driver','N/A'), p.get('Team','N/A'), prob))
    res.sort(key=lambda x:x[2], reverse=True)
    return f"Predictions for {grand_prix_input}, {year}:\n" + "\n".join([f"{i}. {d} ({t}): {p:.2%}" for i,(d,t,p) in enumerate(res[:5],1)])

# ==== 10. MAIN ====
if __name__ == '__main__':
    gv = GRADIO_VARS
    winners_df, drivers_raw_df, teams_raw_df, fastest_laps_raw_df = load_and_clean_data(True)
    # Default previous position
    def_prev_pos_drv = (int(drivers_raw_df['Pos'].max(skipna=True))+5) if 'Pos' in drivers_raw_df and not drivers_raw_df['Pos'].dropna().empty else 50
    def_prev_pos_team = (int(teams_raw_df['Pos'].max(skipna=True))+5) if 'Pos' in teams_raw_df and not teams_raw_df['Pos'].dropna().empty else 20
    gv["default_prev_pos_driver"] = def_prev_pos_drv
    gv["default_prev_pos_team"] = def_prev_pos_team

    drivers_featured_df = engineer_features(drivers_raw_df, teams_raw_df, fastest_laps_raw_df, def_prev_pos_drv, def_prev_pos_team)
    modeling_df = restructure_for_modeling(winners_df, drivers_featured_df, def_prev_pos_drv, def_prev_pos_team)
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    if len(unique_race_ids) < 2:
        train_df, test_df = modeling_df.copy(), modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=SEED, shuffle=True)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    # 欄位補齊與NaN處理
    train_df = fill_missing(train_df, CAT_COLS, NUM_COLS)
    test_df = fill_missing(test_df, CAT_COLS, NUM_COLS)
    # Drop race_id
    for df_ in [train_df, test_df]:
        if 'race_id' in df_.columns: df_.drop(columns=['race_id'], inplace=True)
    # 預處理
    train_df, test_df, gv["label_encoders"], gv["scaler"], cat_feat_dims_map, gv["cat_cols_ordered"], gv["num_cols_ordered"] = \
        preprocess_data(train_df, test_df, CAT_COLS, NUM_COLS)
    train_ds = F1Dataset(train_df, gv["cat_cols_ordered"], gv["num_cols_ordered"], TARGET_COL)
    test_ds = F1Dataset(test_df, gv["cat_cols_ordered"], gv["num_cols_ordered"], TARGET_COL)
    # Weighted Sampler
    weights = [1./(train_df[TARGET_COL].value_counts().get(t,1)+1e-6) for t in train_df[TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    train_loader = DataLoader(train_ds, batch_size=256, sampler=sampler)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
    # Model & Train
    ordered_cat_dims = [cat_feat_dims_map[col] for col in gv["cat_cols_ordered"]]
    num_num_feats = len(gv["num_cols_ordered"])
    gv["model"] = F1DNN(ordered_cat_dims, num_num_feats).to(DEVICE)
    TRAIN_FLAG = not os.path.exists(MODEL_PATH)
    if TRAIN_FLAG:
        train_model(gv["model"], train_loader, test_loader, n_epoch=50, patience=10)
    # Final Eval
    evaluate_model(gv["model"], test_loader, model_path=MODEL_PATH if not TRAIN_FLAG else None)
    # Gradio input dropdowns
    for yr, grp in drivers_featured_df.groupby('year'): gv["driver_info_by_year"][yr] = grp.to_dict('records')
    gv["all_years"] = sorted(list(drivers_featured_df['year'].unique()))
    gv["all_gps"] = sorted(list(winners_df['Grand Prix'].unique()))
    # Gradio
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# F1 Grand Prix Winner Probability Predictor")
        with gr.Row():
            year_dd = gr.Dropdown(label="Year", choices=gv["all_years"], value=gv["all_years"][-1])
            gp_dd = gr.Dropdown(label="Grand Prix", choices=gv["all_gps"], value=gv["all_gps"][0])
        predict_btn = gr.Button("Predict Probabilities")
        output_tb = gr.Textbox(label="Predicted Win Probabilities (Top 10)", lines=12, interactive=False)
        predict_btn.click(gradio_predict_winner_probabilities, inputs=[year_dd, gp_dd], outputs=[output_tb])
        with gr.Accordion("Training Information", open=False):
            if os.path.exists("./data/loss_curve_final.png"):
                gr.Image(value="./data/loss_curve_final.png", label="Loss Curve")
            else:
                gr.Markdown("Loss curve image not found.")
    demo.launch(share=False)



Accuracy: 0.8671
               precision    recall  f1-score   support

Not Winner(0)       0.98      0.88      0.93      4710
    Winner(1)       0.20      0.68      0.31       220

     accuracy                           0.87      4930
    macro avg       0.59      0.78      0.62      4930
 weighted avg       0.95      0.87      0.90      4930

Confusion Matrix:
[[4125  585]
 [  70  150]]
* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\lu050\anaconda3\envs\pytorch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with f